## Setup

In [1]:
import sys
sys.path.insert(0, "../../src/python/")

In [2]:
import os
import pickle
import random
import warnings
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms
from torchvision.models import convnext_tiny
from torchvision.models import ConvNeXt_Tiny_Weights
from torchvision.transforms import v2

from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

/home/shadeform/miniconda/envs/skin-lesion-notebooks/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Binary Classification Model

In [3]:
TRAIN_IMAGE_DIR = "/home/shadeform/data/train/"
VAL_IMAGE_DIR   = "/home/shadeform/data/test/"

TRAIN_LABEL_FILE="/home/shadeform/data/labels/binary_train.pkl"
VAL_LABEL_FILE="/home/shadeform/data/labels/binary_val.pkl"

IMAGE_SIZE=300
BATCH_SIZE=32
NUM_CLASSES=2
EPOCHS=30
LR=5e-5
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

cuda


## Transforms

In [4]:
weights=ConvNeXt_Tiny_Weights.DEFAULT

train_transform=transforms.Compose([transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8,1.0)), transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
                                    transforms.RandomRotation(20), transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                                    transforms.ToTensor(), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

val_transform=transforms.Compose([transforms.Resize((IMAGE_SIZE,IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])

## Dataset

In [5]:
class BinaryDataset(Dataset):

    def __init__(self,image_dir,mapping_file,transform=None):
        self.image_dir=image_dir
        self.transform=transform
        with open(mapping_file,"rb") as f:
            self.labels=pickle.load(f)
        self.image_names=list(self.labels.keys())
        print("Images :",len(self.image_names))

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self,idx):
        image_name=self.image_names[idx]

        if "." in image_name:
            filename=image_name
        else:
            filename=image_name+".jpg"

        image_path=os.path.join(self.image_dir,filename)
        if not os.path.exists(image_path):
            filename=image_name+".png"
            image_path=os.path.join(self.image_dir,filename)
        image=Image.open(image_path).convert("RGB")
        if self.transform is not None:
            image=self.transform(image)
        label=int(self.labels[image_name])
        return image,label

## Dataset Objects

In [6]:
train_dataset=BinaryDataset(TRAIN_IMAGE_DIR, TRAIN_LABEL_FILE, train_transform)
val_dataset=BinaryDataset(VAL_IMAGE_DIR, VAL_LABEL_FILE, val_transform)

print(len(train_dataset))
print(len(val_dataset))

Images : 30931
Images : 8238
30931
8238


## Dataloaders

In [7]:
train_loader=DataLoader(train_dataset,batch_size=BATCH_SIZE, shuffle=True, num_workers=16, pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=BATCH_SIZE, shuffle=False, num_workers=16, pin_memory=True)

print("Train Images :",len(train_dataset))
print("Validation Images :",len(val_dataset))
print("Train Batches :",len(train_loader))
print("Validation Batches :",len(val_loader))

cutmix = v2.CutMix(num_classes=NUM_CLASSES, alpha=1.0)
mixup = v2.MixUp(num_classes=NUM_CLASSES, alpha=0.4)
mixupcutmix = v2.RandomChoice([cutmix, mixup])

Train Images : 30931
Validation Images : 8238
Train Batches : 967
Validation Batches : 258


## Model

In [8]:
weights = ConvNeXt_Tiny_Weights.DEFAULT
model = convnext_tiny(weights=weights)
in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)

model = model.to(DEVICE)
print(model)

ConvNeXt(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
    )
    (1): Sequential(
      (0): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=96, out_features=384, bias=True)
          (4): GELU(approximate='none')
          (5): Linear(in_features=384, out_features=96, bias=True)
          (6): Permute()
        )
        (stochastic_depth): StochasticDepth(p=0.0, mode=row)
      )
      (1): CNBlock(
        (block): Sequential(
          (0): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (1): Permute()
          (2): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (3): Linear(in_features=

## Loss and Optimizer

In [9]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total Parameters     : {total_params:,}")
print(f"Trainable Parameters : {trainable_params:,}")

images, labels = next(iter(train_loader))
images = images.to(DEVICE)
outputs = model(images)

print("Input Shape :", images.shape)
print("Output Shape :", outputs.shape)

Total Parameters     : 27,821,666
Trainable Parameters : 27,821,666
Input Shape : torch.Size([32, 3, 300, 300])
Output Shape : torch.Size([32, 2])


## Training Function

In [10]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(loader):

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        # Apply MixUp/CutMix to 80% of batches
        if random.random() < 0.8:
            images, labels = mixupcutmix(images, labels)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        # Handle both normal labels and one-hot labels
        if labels.ndim == 2:
            hard_labels = labels.argmax(dim=1)
        else:
            hard_labels = labels
        correct += predicted.eq(hard_labels).sum().item()
        total += hard_labels.size(0)
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

In [11]:
def validate(model, loader, criterion):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = outputs.max(1)

            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels

## Training Loop

In [12]:
best_accuracy = 0
patience = 10
counter = 0
history_train_loss = []
history_val_loss = []
history_train_acc = []
history_val_acc = []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)

    val_loss, val_acc, _, _ = validate(model, val_loader, criterion)
    scheduler.step(epoch + 1)

    history_train_loss.append(train_loss)
    history_val_loss.append(val_loss)

    history_train_acc.append(train_acc)
    history_val_acc.append(val_acc)

    print(f"\nTrain Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc*100:.2f}%")

    print()

    print(f"Validation Loss : {val_loss:.4f}")
    print(f"Validation Accuracy : {val_acc*100:.2f}%")

    if val_acc > best_accuracy:
        best_accuracy = val_acc
        counter = 0
        os.makedirs("saved_models/convnext_binary", exist_ok=True)
        torch.save(model.state_dict(), "saved_models/convnext_binary/best_convnext_binary.pth")
        print("\nBest model saved.")

    else:
        counter += 1
        print(f"\nEarly Stopping Counter : {counter}")
    if counter >= patience:
        print("\nEarly stopping")
        break


Epoch 1/30


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:19<00:00, 13.18it/s]



Train Loss      : 0.5196
Train Accuracy  : 79.35%

Validation Loss : 0.5065
Validation Accuracy : 79.17%

Best model saved.

Epoch 2/30


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:19<00:00, 13.17it/s]



Train Loss      : 0.4829
Train Accuracy  : 82.91%

Validation Loss : 0.5351
Validation Accuracy : 76.68%

Early Stopping Counter : 1

Epoch 3/30


...


Train Loss      : 0.4217
Train Accuracy  : 88.56%

Validation Loss : 0.5687
Validation Accuracy : 77.62%

Early Stopping Counter : 8

Epoch 10/30


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:12<00:00, 20.92it/s]



Train Loss      : 0.4068
Train Accuracy  : 89.96%

Validation Loss : 0.5910
Validation Accuracy : 76.75%

Early Stopping Counter : 9

Epoch 11/30


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:11<00:00, 21.73it/s]


Train Loss      : 0.3937
Train Accuracy  : 91.01%

Validation Loss : 0.6115
Validation Accuracy : 76.86%

Early Stopping Counter : 10

Early stopping


## Best Model

In [13]:
os.makedirs("saved_models/convnext_binary", exist_ok=True)
model.load_state_dict(torch.load("saved_models/convnext_binary/best_convnext_binary.pth", map_location=DEVICE))

<All keys matched successfully>

## Evaluate

In [14]:
model.eval()
predictions = []
targets = []
with torch.no_grad():

    for images, labels in tqdm(val_loader):
        images = images.to(DEVICE)
        outputs = model(images)
        _, preds = outputs.max(1)
        predictions.extend(preds.cpu().numpy())
        targets.extend(labels.numpy())

100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 258/258 [00:12<00:00, 20.51it/s]


## Classification Report and Confusion Matrix

In [15]:
print("Validation Accuracy")
print(accuracy_score(targets, predictions))
print(classification_report(targets, predictions, target_names=["Benign", "Malignant"]))

cm = confusion_matrix(targets, predictions)
print(cm)

Validation Accuracy
0.7916970138383103
              precision    recall  f1-score   support

      Benign       0.71      0.82      0.76      3350
   Malignant       0.86      0.77      0.81      4888

    accuracy                           0.79      8238
   macro avg       0.79      0.80      0.79      8238
weighted avg       0.80      0.79      0.79      8238

[[2743  607]
 [1109 3779]]


## Saving the Model

In [18]:
os.makedirs("saved_models/convnext_binary", exist_ok=True)

torch.save({"model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "best_accuracy": best_accuracy, "epochs": EPOCHS},
            "saved_models/convnext_binary/convnext_binary_checkpoint.pth")

In [20]:
import boto3
BUCKET_NAME = "skin-lesion-data-bucket"
MODEL_FOLDER = "saved_models/convnext_binary"
s3 = boto3.client("s3")

files_to_upload = {"saved_models/convnext_binary/convnext_binary_checkpoint.pth": f"{MODEL_FOLDER}/convnext_binary_checkpoint.pth",}

for local_file, s3_key in files_to_upload.items():
    s3.upload_file(local_file, BUCKET_NAME, s3_key)
    print(f"Uploaded -> s3://{BUCKET_NAME}/{s3_key}")

Uploaded -> s3://skin-lesion-data-bucket/saved_models/convnext_binary/convnext_binary_checkpoint.pth
